In [ ]:
import tifffile
import numpy as np
from pathlib import Path

# --- 1. CONFIGURATION ---
# Adjust these paths to the exact folders where your MSSR images are located
dir_c1 = Path(r"H:\My Drive\FRET Experiments\ANALYSIS\Data for graphs\For DT\61W\LB\8 h\analysis\C1_Clover\MSSR")
dir_c2 = Path(r"H:\My Drive\FRET Experiments\ANALYSIS\Data for graphs\For DT\61W\LB\8 h\analysis\C2_Ruby\MSSR")

# This will create a folder called "Merged_16bit_For_JACoP" at the same level as your C1 and C2 folders
output_dir = dir_c1.parent.parent / "Merged_16bit_For_JACoP"
output_dir.mkdir(exist_ok=True)

# --- 2. SAFE CONVERSION FUNCTION ---
def convert_16bit_without_scaling(img_float32):
    # Clip values below 0 and above 65535 to prevent wrap-around aberrations
    img_clipped = np.clip(img_float32, 0, 65535)
    # Round and convert to integer while preserving the original absolute intensity
    return np.round(img_clipped).astype(np.uint16)

# --- 3. PROCESSING ---
# Search for files with the suffix we left in the MSSR code
c1_files = list(dir_c1.glob("*_c1_MSSR.tif")) 
print(f"Detected {len(c1_files)} C1 files. Starting merge for JACoP...\n")

for c1_path in c1_files:
    # Look for the exact pair by replacing "c1" with "c2" in the filename
    c2_filename = c1_path.name.replace("_c1_MSSR.tif", "_c2_MSSR.tif")
    c2_path = dir_c2 / c2_filename
    
    if c2_path.exists():
        # Read the float32 files
        img_c1 = tifffile.imread(c1_path)
        img_c2 = tifffile.imread(c2_path)
        
        # Convert to 16-bit safely (no Min-Max scaling)
        c1_16b = convert_16bit_without_scaling(img_c1)
        c2_16b = convert_16bit_without_scaling(img_c2)
        
        # Merge channels (Clover = Channel 1, Ruby = Channel 2)
        merged_stack = np.stack([c1_16b, c2_16b], axis=0)
        
        # Name the output file
        output_name = output_dir / c1_path.name.replace("_c1_MSSR.tif", "_MERGED_16BIT.tif")
        
        # Save while preserving metadata for Fiji/ImageJ
        tifffile.imwrite(output_name, merged_stack, imagej=True, metadata={'axes': 'CYX'})
        print(f"  [OK] Saved: {output_name.name}")
    else:
        print(f"  [⚠️] NOT FOUND in C2: {c2_filename}")

print("\n====================================================")
print("PROCESS COMPLETED! Images are ready for JACoP.")
print("====================================================")